In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"
bronzeSchema = f"bronze_schema_{env}"

targetTable = f"{catalog}.{bronzeSchema}.rawSales"
# print(tablName)
# srcFileLoc=f"s3://saleslake-748005667258-eu-north-1-an/saleslake/{env}/source_files/discount/"
srcFileLoc=f"/Volumes/{catalog}/{bronzeSchema}/vol_saleslake_src_files_{env}/daily_sales/"
print(f"Target table : {targetTable}")
print(f"Source files : {srcFileLoc}")

In [0]:
spark.sql(f"""
          COPY INTO {targetTable}
        FROM (
            SELECT
                sale_id,
                product_id,
                store_id,
                customer_id,
                region_id,
                quantity,
                unit_price,
                gross_amount,
                discount_code,
                discount_amount,
                net_amount,
                cost_amount,
                sale_date,
                payment_method,
                channel,
                current_timestamp() AS ingest_ts
            FROM '{srcFileLoc}'
        )
        FILEFORMAT = CSV
        FORMAT_OPTIONS (
            'header'='true'
        )
        COPY_OPTIONS (
            'mergeSchema'='false'
        )
    """)    
print("Sales Bronze Load Successful")


In [0]:
spark.sql(f"""
        SELECT COUNT(*) AS total_records
        FROM {targetTable}
    """).show()